In [18]:
import torch
import torchvision
from torchvision import datasets
from pathlib import Path 
from torch.utils.data import DataLoader
import os
from tqdm.auto import tqdm
import wandb

In [4]:
vit_weights = torchvision.models.ViT_B_32_Weights.DEFAULT
vit_transforms = vit_weights.transforms()

In [5]:
vit_data_dir = Path("vit_data")

vit_train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.TrivialAugmentWide(),
    vit_transforms 
])

# Getting ViT traning data
vit_train_data = datasets.FGVCAircraft(root=vit_data_dir,
                                       split="train",
                                       transform=vit_train_transforms,
                                       download=True)

# Get ViT test data
vit_test_data = datasets.FGVCAircraft(root=vit_data_dir,
                                      split="test",
                                      transform=vit_transforms,
                                      download=True)

In [8]:
NUM_WORKERS = os.cpu_count()
BATCH_SIZE = 64

vit_train_dataloder = DataLoader(dataset=vit_train_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

vit_test_dataloder = DataLoader(dataset=vit_test_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

In [10]:
num_classes = len(vit_train_data.classes)
num_classes

100

In [11]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [12]:
from torch import nn

def create_vit(device,
               num_classes: int = 100,
               seed: int = 42):
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    model = torchvision.models.vit_b_16(weights=weights).to(device)

    # Freeze feature extraction layers
    for param in model.parameters():
        param.requires_grad = False

    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    # Create a new classifier layer
    
    model.heads = nn.Sequential(
        nn.Linear(in_features=768, out_features=num_classes)
    )

    return model

In [19]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          learning_rate,
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    wandb.init(project="AircraftDetection", config={"epochs": epochs, "model name":model_name, "learning rate": learning_rate})

    for epoch in tqdm(range(epochs)):
        # Put the model in train mode
        model.to(device)
        model.train()

        # Setup train loss and train accuracy values
        train_loss, train_acc = 0, 0

        for batch, (X, y) in enumerate(train_dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            y_pred = model(X)

            # Calculate and accumulate loss 
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Optimizer zero grad
            optimizer.zero_grad()

            # Loss backward
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Calculate and accumulate accuray metrics across all batches
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item()/len(y_pred)

        # Adjust metrics to get average loss and accuracy per batch
        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)

        # Put the model in eval mode
        model.to(device)
        model.eval()

        # Setup test loss and test accuracy values
        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            # Loop through DataLoader batches
            for batch, (X, y) in enumerate(test_dataloader):
                # Send data to target deivce
                X, y = X.to(device), y.to(device)

                # Forward pass
                test_pred_logits = model(X)

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

            # Adjust metrics to get average loss and accuracy per batch
            test_loss = test_loss / len(train_dataloader)
            test_acc = test_acc / len(test_dataloader)
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            wandb.log({"test_loss": test_loss, "test_acc": test_acc, "train_loss": train_loss, "train_acc": train_acc})

    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        return results
    else:
        return results

In [20]:
lr = 0.1

vit_75_epochs = create_vit(device=device, num_classes=num_classes, seed=42)

vit_75_epochs.to(device)

vit_75_epochs_loss_fn = torch.nn.CrossEntropyLoss()

vit_75_epochs_optimizer = torch.optim.Adam(params=vit_75_epochs.parameters(),
                             lr=lr)

train(model=vit_75_epochs,
      train_dataloader=vit_train_dataloder,
      test_dataloader=vit_test_dataloder,
      loss_fn=vit_75_epochs_loss_fn,
      learning_rate=lr,
      optimizer=vit_75_epochs_optimizer,
      device=device,
      epochs=75,
      save_model=True,
      save_model_path="./models",
      model_name="vit_75_epochs")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: pypdeveloper. Use `wandb login --relogin` to force relogin


  0%|          | 0/75 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 32.8487 | train_acc: 0.0969 | test_loss: 16.3504 | test_acc: 0.1769
Epoch: 2 | train_loss: 12.5572 | train_acc: 0.2452 | test_loss: 14.8258 | test_acc: 0.1960
Epoch: 3 | train_loss: 11.8728 | train_acc: 0.3316 | test_loss: 11.7583 | test_acc: 0.2490
Epoch: 4 | train_loss: 9.8876 | train_acc: 0.3886 | test_loss: 12.9056 | test_acc: 0.2576
Epoch: 5 | train_loss: 9.4777 | train_acc: 0.4280 | test_loss: 11.7563 | test_acc: 0.2782
Epoch: 6 | train_loss: 9.3637 | train_acc: 0.4455 | test_loss: 12.9208 | test_acc: 0.2965
Epoch: 7 | train_loss: 8.6569 | train_acc: 0.4797 | test_loss: 15.9153 | test_acc: 0.2463
Epoch: 8 | train_loss: 9.8955 | train_acc: 0.4811 | test_loss: 14.9215 | test_acc: 0.2953
Epoch: 9 | train_loss: 9.3158 | train_acc: 0.4887 | test_loss: 16.8630 | test_acc: 0.2649
Epoch: 10 | train_loss: 10.2869 | train_acc: 0.5279 | test_loss: 13.9071 | test_acc: 0.3245
Epoch: 11 | train_loss: 8.6162 | train_acc: 0.5628 | test_loss: 14.1635 | test_acc: 0.3398
Epoc

python(49945) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49946) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49947) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49948) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49949) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49950) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49951) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49952) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49953) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49954) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch: 25 | train_loss: 7.4791 | train_acc: 0.6707 | test_loss: 17.9975 | test_acc: 0.3504


...

Epoch: 74 | train_loss: 8.2692 | train_acc: 0.7455 | test_loss: 26.9772 | test_acc: 0.3673


python(60087) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60088) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60089) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60090) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60092) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60093) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60094) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60095) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60096) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60097) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(60150) Malloc

Epoch: 75 | train_loss: 9.0245 | train_acc: 0.7651 | test_loss: 23.7065 | test_acc: 0.4003
[INFO] Saving vit_75_epochs model to ./models


{'train_loss': [9.024471341439014],
 'train_acc': [0.7651336477987422],
 'test_loss': [23.706539513929833],
 'test_acc': [0.40029481132075473]}

wandb: Network error (ConnectionError), entering retry loop.
